# 01 — Pretrain LocalMind

Thin launcher for the SS8 Phase 4 pretraining run on 2x T4.

**Budget:** ~7 h for the 1.5B-token 31M run; 30 GPU-h/week quota; 12 h hard session cap.
**Before running:** set `REPO` below and add `HF_TOKEN` to Kaggle Secrets.


In [ ]:
# Thin launcher: clone, install, run. No project logic lives in this notebook.
import subprocess, sys, os, pathlib
REPO = 'https://github.com/YOUR_USERNAME/localmind.git'
WORK = pathlib.Path('/kaggle/working/localmind')
if not WORK.exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(WORK)], check=True)
os.chdir(WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[torch,tok,data]'], check=True)
import torch; print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),
      '|', torch.cuda.device_count(),'x', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# T4 is SM 7.5: fp16 + GradScaler only. Fail loudly if someone puts bf16 in a config.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('compute capability', cap)
    assert not torch.cuda.is_bf16_supported() or cap[0] >= 8, 'bf16 claim on pre-Ampere'
    print('bf16 supported:', torch.cuda.is_bf16_supported(), '-> plan mandates fp16 regardless')


In [ ]:
# Session-cap insurance (SS3.2 item 3): push checkpoints to HF Hub every hour.
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['LOCALMIND_HUB_REPO'] = 'YOUR_HF_USERNAME/localmind-31m'


## Smoke test first — SS4 DoD says this must work with zero edits


In [ ]:
!python -m localmind.train.loop --config configs/train/smoke.yaml


## Main pretrain (2x T4, DDP)
Resumes bit-exactly if the session is killed — just re-run this cell.


In [ ]:
!torchrun --nproc_per_node=2 -m localmind.train.loop \n    --config configs/train/pretrain.yaml --resume auto
